# 03 — Spearman correlation network (descriptive, not used for model selection)

Pairwise Spearman correlations among the 29 primary biomarkers with Benjamini-Hochberg FDR correction (q < 0.05, |rho| ≥ 0.15). Reported as a descriptive marginal-correlation network alongside the conditional-dependency (Graphical LASSO) network built in stage 04.

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT / "outputs").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if _in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

fd = pd.read_csv(PROJECT_ROOT / 'outputs' / '01_ingest_harmonize' / 'feature_dictionary_v2.csv')
primary = fd.loc[fd.included_primary, 'feature'].tolist()
Y = pd.read_csv(PROJECT_ROOT / 'outputs' / '02_primary_network_dataset' / 'primary_direct_features_unimputed.csv')

cols = primary
p = len(cols)
rho = np.eye(p); pv = np.ones((p, p)); n = np.zeros((p, p), int); rows = []
for i in range(p):
    for j in range(i + 1, p):
        pair = Y[[cols[i], cols[j]]].dropna()
        n[i, j] = n[j, i] = len(pair)
        if len(pair) >= 30:
            r, pp = stats.spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
            rho[i, j] = rho[j, i] = r
            pv[i, j] = pv[j, i] = pp
            rows.append({'source': cols[i], 'target': cols[j], 'rho': r, 'p': pp, 'n': len(pair),
                         'source_domain': DOM[cols[i]], 'target_domain': DOM[cols[j]]})

q = multipletests([r['p'] for r in rows], method='fdr_bh')[1] if rows else []
for k, val in enumerate(q):
    rows[k]['q_fdr'] = val
    rows[k]['abs_rho'] = abs(rows[k]['rho'])
    rows[k]['cross_domain'] = rows[k]['source_domain'] != rows[k]['target_domain']

edges_s = pd.DataFrame(rows).query('q_fdr < 0.05 and abs_rho >= 0.15').sort_values('abs_rho', ascending=False)

out_dir = PROJECT_ROOT / 'outputs' / '03_correlation_network'
out_dir.mkdir(parents=True, exist_ok=True)
edges_s.to_csv(out_dir / 'spearman_edges_fdr_q05_abs015_primary.csv', index=False)
pd.DataFrame(rho, index=cols, columns=cols).to_csv(out_dir / 'spearman_rho_matrix_primary.csv')

print(f'Significant Spearman edges (FDR q<0.05, |rho|>=0.15): {len(edges_s)}')
edges_s.head(10)
